# Qwen-Image-2.1 on Colab: ComfyUI backend for the GIF creator

This notebook turns a Colab GPU into the inference backend for the local Gradio app in this repo:

1. Install ComfyUI (Qwen-Image-2.1 needs **ComfyUI 0.37.0 or newer**).
2. Download the official Qwen-Image-2.1 weights from [`Comfy-Org/Qwen-Image-2.1`](https://huggingface.co/Comfy-Org/Qwen-Image-2.1).
3. Start ComfyUI and expose it through a Cloudflare quick tunnel.
4. Paste the tunnel URL into the app's **Backend Configuration** panel.

> The model weights are released under the **Qwen Research License**. Check its terms before any commercial use.

## 1. Check the GPU
Choose a GPU runtime (*Runtime → Change runtime type*). The int8 weights total roughly 16 GB (diffusion model + text encoder), so an **L4 or A100** is the comfortable choice. On a T4, ComfyUI offloads to CPU RAM automatically: it can work, but it is slow.

In [ ]:
!nvidia-smi

## 2. Install ComfyUI

In [ ]:
!git clone https://github.com/comfyanonymous/ComfyUI.git
%cd ComfyUI
!pip install -q -r requirements.txt

## 3. Download the Qwen-Image-2.1 weights
These are the int8 files used by the official ComfyUI edit template, and the same names `workflow_api.json` expects. For higher quality on large GPUs, swap in the `*_bf16` variants and update the file names in `workflow_api.json`.

In [ ]:
BASE = "https://huggingface.co/Comfy-Org/Qwen-Image-2.1/resolve/main"
!wget -q -nc -P models/diffusion_models {BASE}/diffusion_models/qwen_image_2.1_int8_convrot.safetensors
!wget -q -nc -P models/text_encoders   {BASE}/text_encoders/qwen3vl_8b_int8_convrot.safetensors
!wget -q -nc -P models/vae             {BASE}/vae/qwen_image_2.1_vae_bf16.safetensors
!ls -lh models/diffusion_models models/text_encoders models/vae

## 4. Start ComfyUI and open a tunnel

In [ ]:
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null

import subprocess
import time

subprocess.Popen(["python", "main.py", "--listen", "0.0.0.0", "--port", "8188"])
time.sleep(10)  # give ComfyUI time to boot

# Prints a public https://<random>.trycloudflare.com URL. Keep this cell running.
!cloudflared tunnel --url http://127.0.0.1:8188

## 5. Connect the local app
On your machine, run the app (`uv run python main.py`), open **Backend Configuration & Server Status**, paste the `trycloudflare.com` URL and press **Test Connection**.

Then:
1. Upload a base image (it stays fixed as `<image1>` for every frame).
2. Write an edit instruction, e.g. *"<image1> Turn the head slightly to the right, keep lighting and background unchanged"*.
3. Accept the frames you like and compile the GIF.

> Anyone with the tunnel URL can reach this ComfyUI instance while the cell runs. Stop the cell when you are done.